### Aravec Embeddings

Loading Aravect

In [71]:
import re
import math
import random
import numpy as np
import pandas as pd
from collections import Counter
from typing import List, Dict, Tuple, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import gensim

In [72]:
import pathlib as path
import gensim

model_path = path.Path(r"C:\\ghoufrane\\Master\\M2\\llms\\Hanwritten-code-For-LLM-FineTuning\\models\\AraVec\\tweets_sg_100")
aravec_model = gensim.models.Word2Vec.load(str(model_path))

train = pd.read_csv(r'..\\data\\cleaned_splits\\processed_train.csv')
val = pd.read_csv(r'..\\data\\cleaned_splits\\processed_val.csv')

In [73]:
url_test = "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/testing_data.csv"
test = pd.read_csv(url_test)
print(f"Test size:  {len(test)}")

Test size:  3000


In [74]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {DEVICE}")

Utilisation du device : cpu


Le taux d'OOV (Out-Of-Vocabulary Rate) est le pourcentage de mots inconnus présents dans un nouvel échantillon de texte (ensemble de test) par rapport au vocabulaire de référence utilisé. 

### Tokenization

In [75]:
def tokenize(text):
    if not isinstance(text,str):
        return []
    return text.strip().split()

train['tokens'] = train['tweet'].apply(tokenize)
val['tokens'] = val['tweet'].apply(tokenize)
test['tokens'] = test['tweet'].apply(tokenize)

train[['tweet', 'tokens']].head(3)

,tweet,tokens
0,عاصمه الامويين طاعون اللطميات الشيعيه يغزو...,"[عاصمه, الامويين, طاعون, اللطميات, الشيعيه, يغ..."
1,مش صالح دا كنتوا مخبينه من شعبه,"[مش, صالح, دا, كنتوا, مخبينه, من, شعبه]"
2,رد علي سءال برلماني ابل مشروع جنوب عبدالله ال...,"[رد, علي, سءال, برلماني, ابل, مشروع, جنوب, عبد..."


In [76]:
all_train_words = [word for tokens in train['tokens'] for word in tokens]
word_counts = Counter(all_train_words)

word2idx = {
    '<PAD>': 0,
    '<UNK>': 1
}

MIN_FREQ = 2
for word, count in word_counts.items():
    if count >= MIN_FREQ:
        word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}


print(f"Nombre total de mots uniques dans Train  : {len(word_counts):,}")
print(f"Mots éliminés (fréquence < {MIN_FREQ})        : {sum(1 for c in word_counts.values() if c < MIN_FREQ):,}")
print(f"Mots conservés (fréquence >= {MIN_FREQ})      : {len(word2idx) - 2:,}")
print(f"Taille finale du vocabulaire (+PAD,+UNK) : {len(word2idx):,}")

Nombre total de mots uniques dans Train  : 43,286
Mots éliminés (fréquence < 2)        : 28,305
Mots conservés (fréquence >= 2)      : 14,981
Taille finale du vocabulaire (+PAD,+UNK) : 14,983


### Calcule d'OOV

Token-level OOV = "Quel pourcentage des mots du texte brut sont remplacés par <UNK> ?"

Type-level OOV = "Quel pourcentage des mots du dictionnaire sont inconnus ?"

In [77]:
def get_oov_rates(tokens_series, vocab_dict, split_name="Validation"):
    all_tokens = [w for tokens in tokens_series for w in tokens]
    total_tokens = len(all_tokens)

    oov_tokens = [w for w in all_tokens if w not in vocab_dict or vocab_dict[w] == 1]

    unique_words = set(all_tokens)
    unique_oov_words = set(oov_tokens)

    token_oov_rate = (len(oov_tokens) / total_tokens) * 100 if total_tokens > 0 else 0
    type_oov_rate = (len(unique_oov_words) / len(unique_words)) * 100 if len(unique_words) > 0 else 0

    print(f"--- Taux d'OOV pour {split_name} ---")
    print(f"• Token-level OOV : {token_oov_rate:.2f}% ({len(oov_tokens):,} / {total_tokens:,} mots)")
    print(f"• Type-level OOV  : {type_oov_rate:.2f}% ({len(unique_oov_words):,} / {len(unique_words):,} mots uniques)")
    
    return token_oov_rate, type_oov_rate


val_token_oov, val_type_oov = get_oov_rates(val['tokens'], word2idx, split_name="Validation")

if 'test' in globals() and test is not None:
    test_token_oov, test_type_oov = get_oov_rates(test['tokens'], word2idx, split_name="Test")


--- Taux d'OOV pour Validation ---
• Token-level OOV : 24.91% (4,631 / 18,589 mots)
• Type-level OOV  : 48.67% (4,425 / 9,092 mots uniques)
--- Taux d'OOV pour Test ---
• Token-level OOV : 53.73% (27,166 / 50,556 mots)
• Type-level OOV  : 78.70% (19,450 / 24,715 mots uniques)


In [78]:
def words_to_ids(tokens):
    return [word2idx.get(w, 1) for w in tokens] # 1 est l'index de <UNK>

train['token_ids'] = train['tokens'].apply(words_to_ids)
val['token_ids'] = val['tokens'].apply(words_to_ids)

if 'test' in globals() and test is not None:
    test['token_ids'] = test['tokens'].apply(words_to_ids)

train[['tokens', 'token_ids']].head(3)


,tokens,token_ids
0,"[عاصمه, الامويين, طاعون, اللطميات, الشيعيه, يغ...","[2, 1, 1, 1, 3, 1, 1, 4, 1, 1, 5, 6, 7, 8]"
1,"[مش, صالح, دا, كنتوا, مخبينه, من, شعبه]","[9, 10, 11, 12, 1, 13, 14]"
2,"[رد, علي, سءال, برلماني, ابل, مشروع, جنوب, عبد...","[15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 1, 25..."


### clean_str

In [79]:
import re

def clean_str_aravec(text):
    """
    Fonction officielle AraVec de normalisation de texte arabe
    à appliquer avant la recherche dans aravec_model.wv.
    """
    if not isinstance(text, str):
        return ""
        
    search = ["أ","إ","ء","آ","ة","_","-","/",".","،"," و "," يا ",'"',"ـ","'","ى","\\",'\n', '\t','&quot;','?','؟','!']
    replace = ["ا","ا","ا","ه"," "," ","","",""," و"," يا","","","","ي","",' ', ' ',' ',' ? ',' ؟ ',' ! ']
    
    # 1. Suppression du tashkeel (diacritiques)
    p_tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(p_tashkeel, "", text)
    
    # 2. Réduction des allongements de lettres
    p_longation = re.compile(r'(.)\1+')
    subst = r"\1\1"
    text = re.sub(p_longation, subst, text)
    
    text = text.replace('وو', 'و')
    text = text.replace('يي', 'ي')
    text = text.replace('اا', 'ا')
    
    # 3. Remplacement des caractères
    for s, r in zip(search, replace):
        text = text.replace(s, r)
        
    return text.strip()


### Croisement de notre vocabulaire avec AraVec

In [80]:
vocab_size = len(word2idx)
embed_dim = aravec_model.wv.vector_size  # 100 dimensions
print(f"Vocabulaire : {vocab_size} mots | Dimension des vecteurs AraVec : {embed_dim}")

Vocabulaire : 14983 mots | Dimension des vecteurs AraVec : 100


In [81]:
weights_matrix = np.random.normal(scale=0.02, size=(vocab_size, embed_dim)).astype(np.float32)
weights_matrix[0] = np.zeros(embed_dim, dtype=np.float32)


In [82]:
matched = 0
unmatched = 0
for word, idx in word2idx.items():
    if idx == 0:  # Ignorer <PAD>
        continue
    
    cleaned_word = clean_str_aravec(word)
    
    if cleaned_word in aravec_model.wv:
        weights_matrix[idx] = aravec_model.wv[cleaned_word]
        matched += 1
    elif word in aravec_model.wv:
        # Essai avec le mot non nettoyé au cas où
        weights_matrix[idx] = aravec_model.wv[word]
        matched += 1
    else:
        # Mot absent d'AraVec : il conserve son vecteur aléatoire
        unmatched += 1
coverage = (matched / (vocab_size - 1)) * 100
print(f"\n--- Bilan de la recherche dans AraVec ---")
print(f"• Mots trouvés dans AraVec   : {matched:,} ({coverage:.2f}%)")
print(f"• Mots absents (init hasard) : {unmatched:,} ({100 - coverage:.2f}%)")



--- Bilan de la recherche dans AraVec ---
• Mots trouvés dans AraVec   : 13,979 (93.31%)
• Mots absents (init hasard) : 1,003 (6.69%)


### les mots manquants

In [83]:
missing_words = [word for word, idx in word2idx.items() if idx != 0 and clean_str_aravec(word) not in aravec_model.wv and word not in aravec_model.wv]
print(f"Total mots manquants : {len(missing_words)}")
print("Exemples de mots manquants :", missing_words[:25])


Total mots manquants : 1003
Exemples de mots manquants : ['<UNK>', 'حزب', 'سءال', 'ومصر', 'مصر', '!!', 'هناك', 'كلينتونvsترامب', 'رءيسا', 'مصطفي', 'عااااااجلكافه', 'برشلووونهالتفاصيل', 'مصممه', 'النايك', 'توصوها', 'gymprincesscustomsاللي', 'مصالحه', 'فهءلاء', 'غشششششششاااااام', 'بجاءزه', 'جاءزه', 'سكس', 'قحبه', 'مصابي', 'الغيورون']


In [84]:
weights_tensor = torch.tensor(weights_matrix, dtype=torch.float32)

### freezed

In [85]:
embedding_aravec_figee = nn.Embedding.from_pretrained(
    weights_tensor, 
    freeze=True,       # Bloque la mise à jour (requires_grad = False)
    padding_idx=0      # Garantit que <PAD> reste toujours à zéro
)

### unfreezed

In [87]:
embedding_aleatoire = nn.Embedding(
    num_embeddings=vocab_size, 
    embedding_dim=embed_dim, 
    padding_idx=0
)
nn.init.normal_(embedding_aleatoire.weight, mean=0.0, std=0.1)
embedding_aleatoire.weight.data[0].fill_(0.0) # <PAD> à zéro


tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.])

### aléatoire

In [ ]:
embedding_aleatoire = nn.Embedding(
    num_embeddings=vocab_size, 
    embedding_dim=embed_dim, 
    padding_idx=0
)
nn.init.normal_(embedding_aleatoire.weight, mean=0.0, std=0.1)
embedding_aleatoire.weight.data[0].fill_(0.0) # <PAD> à zéro